# Scheme 1 XGBoost Demo: Saved-Model Inference, Evaluation, Wrong Predictions, Tolerance and Important Failures

**Purpose.** This notebook is a demonstration/inference workflow for the already-trained Scheme 1 XGBoost models using the weighting experiment **`log_time_min_plus_log_time_ratio`**. It does **not retrain** the model.

It supports both trained split variants:
- `strat_y` — target-only stratification
- `strat_y_K` — target + K stratification

The new dataset is expected to have the **same raw schema** as the training dataset, including the 24 algorithm runtime columns. That is necessary because the notebook evaluates ground-truth fastest algorithms, runtime regret, wrong predictions, tolerance, and important failures.

> **Demo principle:** change the dataset path and model paths in the configuration cell; the rest of the notebook reuses the existing pipeline functions.


##  What to change when demonstrating another dataset

For a replacement dataset with the same schema:

1. Change **only `NEW_DATASET_PATH`** in Cell 1.
2. Keep the two trained model paths unchanged unless you want to demonstrate another trained run.
3. Run all cells from the top.
4. Inspect `demo_summary`, tolerance results, wrong predictions, and important failures in the notebook.
5. Collect exported artifacts from `OUTPUT_DIR`.

### Important interpretation
The weighting scheme `log_time_min_plus_log_time_ratio` influenced **how the XGB model was trained**. It is not a second model applied at prediction time. The inference path uses the saved classifier and exactly the feature representation it was trained on. Runtime columns are needed here because this is an **evaluation demo**; if a future production dataset has features but no ground-truth runtimes, prediction still works but accuracy/tolerance/failure analysis cannot be computed.


## Cell 1 — Repository and user inputs

**User edits this cell.**

Set `REPO_ROOT` to the repository root and `NEW_DATASET_PATH` to the replacement dataset. The two model paths are placeholders matching your current folder structure. Replace the `*` with the exact `.pkl` filename when available.

Outputs from this notebook are deliberately placed in a separate `demo_scheme1_log_time_min_plus_log_time_ratio` report folder so the demonstration does not overwrite the original experiment.


In [ ]:
from pathlib import Path
import sys, glob, copy, pickle

REPO_ROOT = Path(r"D:\Studies\Research\Da_Project\kpex_ML_git")
NEW_DATASET_PATH = REPO_ROOT / r"restructured_pipeline\data\PLACEHOLDER_NEW_DATASET.csv"

WEIGHT_SCHEME_NAME = "log_time_min_plus_log_time_ratio"
RUN_TAG = "23sep2026"

MODEL_GLOBS = {
    "strat_y": REPO_ROOT / rf"restructured_pipeline\outputs\models\scheme1\xgboost\{WEIGHT_SCHEME_NAME}\model_scheme1_xgboost_xgb_strat_y*_weight_{WEIGHT_SCHEME_NAME}_{RUN_TAG}.pkl",
    "strat_y_K": REPO_ROOT / rf"restructured_pipeline\outputs\models\scheme1\xgboost\{WEIGHT_SCHEME_NAME}\model_scheme1_xgboost_xgb_strat_y_K*_weight_{WEIGHT_SCHEME_NAME}_{RUN_TAG}.pkl",
}

PROJECT_CONFIG_PATH = REPO_ROOT / "restructured_pipeline/configs/project.yaml"
MODEL_CONFIG_PATH = REPO_ROOT / "restructured_pipeline/configs/models_scheme1.yaml"
OUTPUT_DIR = REPO_ROOT / f"restructured_pipeline/outputs/reports/demo_scheme1_{WEIGHT_SCHEME_NAME}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("Repository:", REPO_ROOT)
print("New dataset:", NEW_DATASET_PATH)
print("Output directory:", OUTPUT_DIR)


## Cell 2 — Import the existing pipeline functions

This notebook intentionally delegates the core logic to your existing Python modules:

- `train.load_yaml_config`, `load_model_artifact`, `build_flat_results` — configuration, saved model artifact and flat 24-class prediction.
- `features.load_excel_data`, `assign_weights`, `process_kcore_list_to_statistics`, `post_process_data` — the same raw-data cleaning and feature engineering used by training.
- `evaluate.evaluate_results`, `compute_top_k_metrics` — classification/runtime/Top-K evaluation.
- `metrics.evaluate_algorithm_performance`, `evaluate_with_tolerance`, `evaluate_time_metrics*` — existing metric definitions.
- `compare_wrong_predictions.filter_tolerance_sheet_for_important_failures` — the existing important-failure rule.

The only notebook-local functions will be orchestration helpers needed to apply the existing preprocessing to a **user-selected dataset path**, because the training helper normally reads the dataset path from `project.yaml`.


In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

from restructured_pipeline.src.train import (
    load_yaml_config,
    load_model_artifact,
    build_flat_results,
    build_wrong_predictions,
    normalize_weight_scheme,
)
from restructured_pipeline.src.features import (
    algorithms as ALL_ALGOS,
    feature_cols as PIPELINE_FEATURES,
    load_excel_data,
    assign_weights,
    process_kcore_list_to_statistics,
    post_process_data,
)
from restructured_pipeline.src.evaluate import evaluate_results, compute_top_k_metrics
from restructured_pipeline.src.metrics import (
    evaluate_algorithm_performance,
    evaluate_time_metrics,
    evaluate_with_tolerance,
    evaluate_time_metrics_with_tol,
)
from restructured_pipeline.src.compare_wrong_predictions import (
    filter_tolerance_sheet_for_important_failures,
)

print(f"Algorithms: {len(ALL_ALGOS)}")
print(f"Pipeline features: {PIPELINE_FEATURES}")


## Cell 3 — Load configuration and verify the experiment definition

The Scheme 1 configuration defines XGBoost as a **24-class `multi:softprob` classifier** for both `xgb_strat_y` and `xgb_strat_y_K`. The saved model artifact is authoritative for the exact feature list, algorithm vocabulary, label encoder and trained classifier.

The weighting formula matters during **training**. At inference, the trained model only consumes the engineered feature vector. We still compute the same weight column while preprocessing because the existing pipeline's dataframe schema contains it and because it keeps the demo preprocessing consistent with the experiment.


In [ ]:
project = load_yaml_config(PROJECT_CONFIG_PATH)
model_cfg = load_yaml_config(MODEL_CONFIG_PATH)

print("Configured Scheme 1 XGB variants:")
for name, cfg in model_cfg["scheme1"]["xgboost"]["variants"].items():
    print(f"  {name}: split_mode={cfg['split_mode']}, num_class={cfg['hyperparameters'].get('num_class')}")

assert len(project["targets"]["scheme1"]["algorithms"]) == 24
assert set(project["targets"]["scheme1"]["algorithms"]) == set(ALL_ALGOS)
print()
print("Scheme 1 algorithm vocabulary verified: 24 algorithms.")


## Cell 4 — Resolve and load the two trained `.pkl` artifacts

Each pickle is expected to be the artifact written by your training pipeline, containing at least:
`classifier`, `label_encoder`, `feature_cols`, `algorithms`, `split_mode`, and model metadata.

If the wildcard finds zero or multiple files, the cell stops rather than guessing which model to use.


In [ ]:
def resolve_single_model(pattern: Path) -> Path:
    matches = [Path(p) for p in glob.glob(str(pattern))]
    if len(matches) != 1:
        raise FileNotFoundError(f"Expected exactly one model for pattern: {pattern}. Found {len(matches)}: {matches}")
    return matches[0]

model_paths = {split: resolve_single_model(pattern) for split, pattern in MODEL_GLOBS.items()}
artifacts = {}

for split, path in model_paths.items():
    artifact, resolved, _ = load_model_artifact(project, "scheme1", "xgboost", base_run_name=path.stem.removeprefix("model_"), model_path=path)
    artifacts[split] = artifact
    print(f"{split}: {resolved}")
    print("  saved split_mode:", artifact.get("split_mode"))
    print("  features:", len(artifact.get("feature_cols", [])))
    print("  algorithms:", len(artifact.get("algorithms", [])))


## Cell 5 — Load and preprocess the new dataset with the existing feature pipeline

This is the one place where we need a small notebook wrapper. `train.load_and_preprocess_data()` reads the data path from `project.yaml`; for a demo dataset we instead pass `NEW_DATASET_PATH` while calling the **same underlying feature functions in the same order**:

1. `load_excel_data()` — CSV parsing, column renaming, timeout replacement, removal of all-timeout rows, numeric runtime conversion.
2. `assign_weights()` — same `log_time_min_plus_log_time_ratio` training weight definition.
3. `process_kcore_list_to_statistics()` — derives `core_min`, shell count and shell-size statistics.
4. `post_process_data()` — applies the established output schema/order.

No new feature engineering is introduced.


In [ ]:
def preprocess_demo_dataset(dataset_path: Path, project_config: dict) -> pd.DataFrame:
    if not dataset_path.exists():
        raise FileNotFoundError(
            f"Set NEW_DATASET_PATH in Cell 1. File does not exist: {dataset_path}"
        )

    data_cfg = project_config["data"]
    eval_cfg = project_config["evaluation"]

    df = load_excel_data(
        dataset_path,
        timeout_replacement=data_cfg["timeout_replacement"],
        y_cols=data_cfg["raw_target_columns"],
        sep=data_cfg["sep"],
        quotechar=data_cfg["quotechar"],
        engine=data_cfg["engine"],
        rename_columns=data_cfg["rename_columns"],
    )

    assign_weights(
        df,
        normalize_weight_scheme(WEIGHT_SCHEME_NAME),
        eval_cfg["epsilon"],
    )
    process_kcore_list_to_statistics(df)
    df = post_process_data(df)
    df["source_row_index"] = df.index
    return df

work = preprocess_demo_dataset(NEW_DATASET_PATH, project)
print("Rows after preprocessing:", len(work))
display(work.head())


## Cell 6 — Hard compatibility checks before inference

A demo should fail loudly if the replacement dataset is incompatible. This cell verifies:

- all 24 runtime columns exist (required for ground-truth evaluation),
- every feature expected by each saved model exists,
- model algorithm vocabulary matches the dataset,
- model feature matrices contain no missing values.

This prevents a superficially successful but invalid prediction run.


In [ ]:
missing_runtime = [a for a in ALL_ALGOS if a not in work.columns]
if missing_runtime:
    raise ValueError(f"Dataset is missing algorithm runtime columns: {missing_runtime}")

for split, artifact in artifacts.items():
    features = artifact.get("feature_cols") or project["features"]["feature_cols"]
    algorithms = artifact.get("algorithms") or project["targets"]["scheme1"]["algorithms"]
    missing_features = [c for c in features if c not in work.columns]
    if missing_features:
        raise ValueError(f"{split}: missing model features: {missing_features}")
    if set(algorithms) != set(ALL_ALGOS):
        raise ValueError(f"{split}: saved model algorithm vocabulary differs from Scheme 1.")
    bad = work[features].isna().sum()
    bad = bad[bad > 0]
    if not bad.empty:
        raise ValueError(f"{split}: missing feature values after preprocessing: {bad.to_dict()}")
    print(f"{split}: compatibility checks passed ({len(features)} features, {len(algorithms)} algorithms).")


## Cell 7 — Run flat 24-algorithm prediction with each saved model

This reuses `train.build_flat_results()`, the same flat prediction path used by Scheme 1. For every row it:

- determines the complete set of tied globally-fastest algorithms,
- calls the trained XGB classifier,
- decodes the predicted class through the saved label encoder,
- writes one `prob_<algorithm>` column for every class,
- marks exact correctness (ties count as correct),
- builds Top-3 predictions,
- derives wrong-prediction runtime information.

The two models are evaluated independently on the **same new dataset**.


In [ ]:
results = {}
wrong = {}
base_metrics = {}

for split, artifact in artifacts.items():
    features = artifact.get("feature_cols") or project["features"]["feature_cols"]
    algorithms = artifact.get("algorithms") or list(ALL_ALGOS)
    clf = artifact["classifier"]
    le = artifact.get("label_encoder")

    cls_metrics, time_metrics, result_df, wrong_df = build_flat_results(
        work.copy(), clf, features, algorithms, le=le,
        epsilon=project["evaluation"]["epsilon"], tol=0.0,
    )
    results[split] = result_df
    wrong[split] = wrong_df
    base_metrics[split] = {"classification": cls_metrics, "time": time_metrics}
    print()
    print(f"{split}: predictions={len(result_df)}, wrong={len(wrong_df)}")



## Cell 8 — Inspect predictions

This is the main demo table. It shows the instance identity, true fastest algorithm set, model prediction, correctness, and the model's Top-3 probability ranking. The full dataframe also retains all engineered features, 24 runtimes, and 24 probability columns.


In [ ]:
for split, df in results.items():
    print()
    print(f"=== {split} ===")
    cols = [c for c in ["dataset", "k_value", "fastest_algorithms", "predicted", "correct", "top3_predicted"] if c in df.columns]
    display(df[cols].head(20))



## Cell 9 — Exact classification and runtime metrics

This reuses `evaluate_algorithm_performance()` and `evaluate_time_metrics()` through `evaluate_results()`.

- **Accuracy:** fraction of rows where the predicted algorithm belongs to the true fastest set.
- **Per-algorithm precision:** when an algorithm is predicted, how often that choice is correct.
- **Per-algorithm recall:** when an algorithm is truly fastest, how often the model recovers it.
- **Time ratio:** total runtime of model-selected algorithms divided by total optimal runtime.

The table below is intentionally compact for a demo; detailed per-algorithm precision/recall are retained separately.


In [ ]:
metric_rows = []
per_algorithm_rows = []

for split, df in results.items():
    algorithms = artifacts[split].get("algorithms") or list(ALL_ALGOS)
    cls, tm, _, _, topk = evaluate_results(
        df, algorithms, tol=0.0, epsilon=project["evaluation"]["epsilon"]
    )
    metric_rows.append({
        "Stratification": split,
        "Rows": len(df),
        "Accuracy": cls["accuracy"],
        "Time Ratio (Predicted / Optimal)": tm["time_ratio"],
        "Top-1 Accuracy": topk["top1_accuracy"],
        "Top-2 Accuracy": topk["top2_accuracy"],
        "Top-3 Accuracy": topk["top3_accuracy"],
    })
    for algo in algorithms:
        per_algorithm_rows.append({
            "Stratification": split,
            "Algorithm": algo,
            "Precision": cls["precision_per_alg"].get(algo, 0.0),
            "Recall": cls["recall_per_alg"].get(algo, 0.0),
        })

metrics_summary = pd.DataFrame(metric_rows)
per_algorithm_metrics = pd.DataFrame(per_algorithm_rows)
display(metrics_summary)
display(per_algorithm_metrics.head(48))


## Cell 10 — Tolerance-aware evaluation (0%, 10%, 20%, 50%)

A prediction can be label-wrong but runtime-near-optimal. Your existing tolerance logic therefore treats any algorithm within `(1 + tolerance) × optimal runtime` as acceptable.

This cell reuses `evaluate_with_tolerance()`, `evaluate_time_metrics_with_tol()`, and `compute_top_k_metrics()` and produces the same four tolerance levels used elsewhere in the pipeline.


In [ ]:
TOLERANCES = [0.0, 0.1, 0.2, 0.5]
tolerance_rows = []

for split, df in results.items():
    algorithms = artifacts[split].get("algorithms") or list(ALL_ALGOS)
    for tol in TOLERANCES:
        cm = evaluate_with_tolerance(df, algorithms, tol=tol)
        tm = evaluate_time_metrics_with_tol(
            df, algorithms, tol=tol, epsilon=project["evaluation"]["epsilon"]
        )
        tk = compute_top_k_metrics(df, algorithms, tol=tol)
        tolerance_rows.append({
            "Stratification": split,
            "Tolerance": tol,
            "Tolerance (%)": int(tol * 100),
            "Accuracy": cm["accuracy"],
            "Time Ratio": tm["time_ratio"],
            "Top-1 Accuracy": tk["top1_accuracy_with_tolerance"],
            "Top-2 Accuracy": tk["top2_accuracy_with_tolerance"],
            "Top-3 Accuracy": tk["top3_accuracy_with_tolerance"],
        })

tolerance_summary = pd.DataFrame(tolerance_rows)
display(tolerance_summary)


## Cell 11 — Wrong predictions and severity information

The existing `build_wrong_predictions()` output is used directly. A wrong prediction includes the optimal runtime, selected runtime, absolute time gap, slowdown ratio, and true fastest algorithm set.

This is the bridge from ordinary classification error to **performance severity**.


In [ ]:
for split, df in wrong.items():
    print()
    print(f"=== Wrong predictions: {split} ({len(df)} rows) ===")
    cols = [c for c in [
        "dataset", "k_value", "fastest_algorithms", "predicted",
        "fastest_time", "predicted_time", "time_gap", "time_ratio", "top3_predicted"
    ] if c in df.columns]
    display(df[cols].head(25))



## Cell 12 — Important failures using the existing pipeline rule

This calls `filter_tolerance_sheet_for_important_failures()` directly. The current pipeline defines an important failure by excluding:

1. cases where **both** optimal and predicted runtimes are below the small-runtime threshold (default 5.0), and
2. cases where predicted runtime is within **1.2×** the optimal runtime.

Therefore, remaining cases are the wrong predictions that are operationally more consequential. No new important-failure definition is introduced in the notebook.


In [ ]:
important_failures = {}
important_stats = {}

for split, wrong_df in wrong.items():
    # The existing filter accepts fastest_time/predicted_time as fallbacks.
    important, stats = filter_tolerance_sheet_for_important_failures(wrong_df)
    important_failures[split] = important
    important_stats[split] = stats
    print()
    print(f"{split}")
    print(stats)
    display(important.head(20))



## Cell 13 — Failure-analysis summary for the demo

This gives the presentation-level counts: total predictions, wrong predictions, important failures, and the important-failure rates relative to both wrong predictions and all predictions.


In [ ]:
failure_rows = []
for split in ("strat_y", "strat_y_K"):
    total = len(results[split])
    n_wrong = len(wrong[split])
    n_important = len(important_failures[split])
    failure_rows.append({
        "Stratification": split,
        "Model": "XGB",
        "Evaluation Scope": "New Dataset",
        "Total Predictions": total,
        "Wrong Predictions": n_wrong,
        "Important Failures": n_important,
        "Important / Wrong (%)": round(100*n_important/n_wrong, 2) if n_wrong else 0.0,
        "Important / All (%)": round(100*n_important/total, 2) if total else 0.0,
    })

failure_summary = pd.DataFrame(failure_rows)
display(failure_summary)


## Cell 14 — Top-1 through Top-6 recovery analysis for important failures

The existing weighted failure-analysis script builds enriched Top-K failure tables through the shared `failure_analysis_summary` utilities. For a self-contained demo on a replacement dataset, this cell computes the same core quantities directly from the already-produced `prob_<algorithm>` columns and runtime columns:

- rank of the true fastest algorithm in the model probability ordering,
- best runtime ratio available within Top-1 … Top-6,
- corresponding candidate,
- complete probability-ranked prediction list.

This notebook-local helper does **not alter model predictions**; it only summarizes them for display.


In [ ]:
def enrich_topk_failure_rows(df: pd.DataFrame, algorithms: list[str]) -> pd.DataFrame:
    rows = []
    for _, r in df.iterrows():
        runtimes = {a: float(r[a]) for a in algorithms}
        fastest_time = min(runtimes.values())
        fastest_algos = [a for a, rt in runtimes.items() if rt == fastest_time]
        ranked = sorted(algorithms, key=lambda a: float(r[f"prob_{a}"]), reverse=True)
        fastest_rank = min(ranked.index(a) + 1 for a in fastest_algos)
        representative_fastest = next(a for a in ranked if a in fastest_algos)

        rec = {
            "dataset": r.get("dataset"),
            "k_value": r.get("k_value"),
            "scheme": "scheme 1",
            "fastest_algo_rank": fastest_rank,
            "fastest_algo_tuple": (fastest_time, representative_fastest, fastest_rank),
        }
        for k in range(1, 7):
            candidates = ranked[:k]
            best = min(candidates, key=lambda a: runtimes[a])
            rec[f"top{k}_min_time_ratio"] = runtimes[best] / fastest_time if fastest_time else np.nan
            rec[f"top{k}_candidate"] = (runtimes[best], best, float(r[f"prob_{best}"]))
        rec["all_predictions_sorted_by_prob"] = [
            (runtimes[a] / fastest_time if fastest_time else np.nan, a, runtimes[a], float(r[f"prob_{a}"]))
            for a in ranked
        ]
        rows.append(rec)
    return pd.DataFrame(rows)

topk_failure_analysis = {}
for split, df in important_failures.items():
    algos = artifacts[split].get("algorithms") or list(ALL_ALGOS)
    topk_failure_analysis[split] = enrich_topk_failure_rows(df, algos)
    print()
    print(f"=== Important-failure Top-K: {split} ===")
    display(topk_failure_analysis[split].head(20))



## Cell 15 — Compact demo comparison

This combines the most useful headline numbers from the two saved models. It is intended for a live demonstration or slide screenshot, while the detailed tables remain available above.


In [ ]:
demo_summary = metrics_summary.merge(
    failure_summary[["Stratification", "Wrong Predictions", "Important Failures", "Important / All (%)"]],
    on="Stratification",
    how="left",
)
display(demo_summary)


## Cell 16 — Export all demo outputs

The notebook exports CSV/XLSX artifacts to the demo output directory rather than the original training experiment directories.

Expected outputs include:
- predictions for each split model,
- wrong predictions,
- important failures,
- Top-K important-failure analysis,
- exact metrics,
- per-algorithm precision/recall,
- tolerance summary,
- failure summary,
- one consolidated Excel workbook for convenient inspection.


In [ ]:
for split in ("strat_y", "strat_y_K"):
    results[split].to_csv(OUTPUT_DIR / f"predictions_{split}.csv", index=False)
    wrong[split].to_csv(OUTPUT_DIR / f"wrong_predictions_{split}.csv", index=False)
    important_failures[split].to_csv(OUTPUT_DIR / f"important_failures_{split}.csv", index=False)
    topk_failure_analysis[split].to_csv(OUTPUT_DIR / f"important_failures_topk_{split}.csv", index=False)

metrics_summary.to_csv(OUTPUT_DIR / "metrics_summary.csv", index=False)
per_algorithm_metrics.to_csv(OUTPUT_DIR / "per_algorithm_precision_recall.csv", index=False)
tolerance_summary.to_csv(OUTPUT_DIR / "tolerance_summary.csv", index=False)
failure_summary.to_csv(OUTPUT_DIR / "failure_summary.csv", index=False)

excel_path = OUTPUT_DIR / "scheme1_xgb_demo_summary.xlsx"
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    demo_summary.to_excel(writer, sheet_name="demo_summary", index=False)
    metrics_summary.to_excel(writer, sheet_name="metrics", index=False)
    per_algorithm_metrics.to_excel(writer, sheet_name="per_algorithm_metrics", index=False)
    tolerance_summary.to_excel(writer, sheet_name="tolerance", index=False)
    failure_summary.to_excel(writer, sheet_name="failure_summary", index=False)
    for split in ("strat_y", "strat_y_K"):
        wrong[split].to_excel(writer, sheet_name=f"wrong_{split}"[:31], index=False)
        important_failures[split].to_excel(writer, sheet_name=f"important_{split}"[:31], index=False)
        topk_failure_analysis[split].to_excel(writer, sheet_name=f"topk_{split}"[:31], index=False)

print("Outputs written to:", OUTPUT_DIR)
print("Consolidated workbook:", excel_path)
